# Urban Mobility Analytics MVP
## Notebook 02: SUBE Transaction & Active Cards Demand Analysis

This notebook analyzes the **temporal patterns of mobility** in the Buenos Aires Metropolitan Area using the SUBE ticketing data, providing key performance indicators (KPIs) to understand demand variations.

### Objectives:
1. **Analyze temporal patterns** (day of week, weekend vs. weekday seasonality).
2. **Calculate Mobility Intensity Metrics** (transactions per active card).
3. **Discuss implications for Origin-Destination (OD) Matrix inference**.
---

### 1. Setup & Data Loading

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

metrics_path = os.path.join('..', config['sube']['mobility_metrics_output'])
df = pd.read_parquet(metrics_path)
df['DIA_TRANSPORTE'] = pd.to_datetime(df['DIA_TRANSPORTE'])

print(f"Loaded {len(df)} days of SUBE activity data.")

### 2. Feature Engineering: Temporal Markers

In [ ]:
# Extract day of the week, month, and type of day (weekday vs. weekend)
df['day_of_week'] = df['DIA_TRANSPORTE'].dt.day_name()
df['day_of_week_num'] = df['DIA_TRANSPORTE'].dt.dayofweek
df['is_weekend'] = df['day_of_week_num'].isin([5, 6])
df['month'] = df['DIA_TRANSPORTE'].dt.to_period('M')

df.head(3)

### 3. Demand Analysis by Day of the Week

In [ ]:
# Grouping transactions by day of the week
weekday_agg = df.groupby(['day_of_week_num', 'day_of_week'])['total_transacciones'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=weekday_agg, x='day_of_week', y='total_transacciones', palette='Blues_d')
plt.title('Average Daily Transactions by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Average Transactions')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 4. Mobility Intensity: Ratio of Transactions per Unique Active Card

In [ ]:
# Distribution of transactions per card (proxy for multi-stage trips)
plt.figure(figsize=(8, 4))
sns.histplot(df['viajes_por_tarjeta'], kde=True, color='purple')
plt.title('Distribution of Mobility Intensity (Transactions per Active Card)')
plt.xlabel('Transactions / Card Ratio')
plt.ylabel('Frequency (Days)')
plt.show()

### 5. Methodological Framework: Origin-Destination Matrix Inference

To transition from aggregate transactional volumes to structural travel flows (Origin-Destination matrices), the following methodologies can be applied if disaggregate transaction logs are available:

1. **Card Chaining Analysis:** Tracking unique, anonymized card identifiers over time to reconstruct sequence journeys.
2. **Destination Inference Algorithms:** Correlating transaction tap-in locations with GTFS schedules to estimate the most likely destination based on subsequent trip start locations.
3. **Flow Aggregation:** Summarizing inferred journeys at zones (e.g., Communes/Barrios) to compile structural Origin-Destination matrices for urban transit optimization.